# nb_01_bronze_ingest
**Milestone 1: Bronze layer.** Loads the 5 raw CSVs from `Files/raw/global_electronics/` into Delta tables **unchanged** (all columns as text), plus audit columns. Metadata-driven: one config list, one loop.

### Cell 1: Parameters

In [ ]:
source_folder = "Files/raw/global_electronics"

### Cell 2: Source configuration
Encoding is set **per file**: Customers.csv is Windows-1252, the others are UTF-8. Spark 4 only accepts a few charsets by default, so legacy Java charsets are enabled.

In [ ]:
import re, uuid
from pyspark.sql import functions as F

# Spark 4 only accepts a few standard charsets by default; allow Java charsets like windows-1252
spark.conf.set("spark.sql.legacy.javaCharsets", "true")

# One entry per source file: what to read, how to read it, and what columns we expect
SOURCES = [
    {"file": "Sales.csv",          "table": "bronze_sales",          "encoding": "UTF-8",
     "columns": ["Order Number","Line Item","Order Date","Delivery Date","CustomerKey",
                 "StoreKey","ProductKey","Quantity","Currency Code"]},
    {"file": "Customers.csv",      "table": "bronze_customers",      "encoding": "windows-1252",
     "columns": ["CustomerKey","Gender","Name","City","State Code","State","Zip Code",
                 "Country","Continent","Birthday"]},
    {"file": "Products.csv",       "table": "bronze_products",       "encoding": "UTF-8",
     "columns": ["ProductKey","Product Name","Brand","Color","Unit Cost USD","Unit Price USD",
                 "SubcategoryKey","Subcategory","CategoryKey","Category"]},
    {"file": "Stores.csv",         "table": "bronze_stores",         "encoding": "UTF-8",
     "columns": ["StoreKey","Country","State","Square Meters","Open Date"]},
    {"file": "Exchange_Rates.csv", "table": "bronze_exchange_rates", "encoding": "UTF-8",
     "columns": ["Date","Currency","Exchange"]},
]

def to_snake(name):
    """'Order Number' -> 'order_number', 'CustomerKey' -> 'customer_key'"""
    name = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", name)       # split CamelCase
    return re.sub(r"[^0-9a-zA-Z]+", "_", name).strip("_").lower()

batch_id = str(uuid.uuid4())
print("Batch:", batch_id)

### Cell 3: Check all files exist before loading anything

In [ ]:
files_found = [f.name for f in notebookutils.fs.ls(source_folder)]
missing_files = [s["file"] for s in SOURCES if s["file"] not in files_found]
if missing_files:
    raise FileNotFoundError(f"Missing in {source_folder}: {missing_files}. Found: {files_found}")
print("All source files present")

### Cell 4: Load loop
Schema contract (fail on missing columns), independent row count from the raw file, snake_case rename (Delta doesn't allow spaces in column names), audit columns, idempotent overwrite, reconciliation.

In [ ]:
results = []
for src in SOURCES:
    path = f"{source_folder}/{src['file']}"

    # 1. Read raw: everything as text, correct encoding, handle quotes inside values (52" LCD)
    df = (spark.read
          .option("header", True)
          .option("inferSchema", False)
          .option("encoding", src["encoding"])
          .option("quote", '"').option("escape", '"')
          .csv(path))

    # 2. Schema contract: fail on missing columns, warn on new ones
    missing = [c for c in src["columns"] if c not in df.columns]
    extra   = [c for c in df.columns if c not in src["columns"]]
    if missing:
        raise ValueError(f"{src['file']}: schema contract broken, missing {missing}")
    if extra:
        print(f"WARNING {src['file']}: unexpected new columns {extra}")

    # 3. Independent row count straight from the raw file (lines minus header)
    file_rows = spark.read.text(path).count() - 1

    # 4. Rename columns + add audit columns
    df_bronze = (df.select([F.col(f"`{c}`").alias(to_snake(c)) for c in df.columns]
                           + [F.col("_metadata.file_path").alias("_source_file")])
                   .withColumn("_ingested_at", F.current_timestamp())
                   .withColumn("_batch_id", F.lit(batch_id)))

    # 5. Write: full overwrite, so reruns never duplicate rows
    (df_bronze.write.format("delta")
        .mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(src["table"]))

    # 6. Reconcile file vs table
    table_rows = spark.table(src["table"]).count()
    status = "PASS" if file_rows == table_rows else "FAIL"
    results.append((src["table"], file_rows, table_rows, status))
    print(f"{src['table']:<24} file={file_rows:>6,}  table={table_rows:>6,}  {status}")

### Cell 5: Summary, and fail the run if anything is off

In [ ]:
summary = spark.createDataFrame(results, ["table", "file_rows", "table_rows", "status"])
display(summary)

failed = [r[0] for r in results if r[3] != "PASS"]
if failed:
    raise Exception(f"Reconciliation FAILED for: {failed}")
print("Bronze load complete: all tables reconciled")